In [2]:
import gc
import torch

gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

import os
os.listdir(); os.chdir("/aiau010_scratch/azm0269/clover/")

from clover.utils.utils import notebook_line_magic
notebook_line_magic()

In [ ]:
# from clover.baselines.md3po import md3po_combined_rollouts


# torch.cuda.is_available()

In [ ]:
# torch.clamp(1-1e-4, torch.tensor([0.9999934786]), 1+1e-4)
ratio = torch.tensor([0.9999934786])
torch.clamp(ratio, 1 - 1e-4, 1 + 1e-4) #* adv

In [12]:
import math
# math.exp(math.log(5.0))
math.log(2)

0.6931471805599453

In [6]:
trajectories = torch.load("/aiau010_scratch/azm0269/clover/clover/data/emo/seed_123/trajectories.pt", 
                          map_location="cpu",
                          weights_only=False)

In [7]:
trajectories.keys()

dict_keys([32])

In [ ]:
# trajectories[1]['rewards'].max()

import matplotlib.pyplot as plt
import math

In [ ]:
from clover.baselines.emo_v2 import leave_one_out_advantages
from clover.baselines.common import discount_rewards, sigmoid_discounting

In [ ]:
beta= 40
rewards = trajectories[32]['rewards'].clone()
prompts = trajectories[32]['prompts']
terminal_advantages = leave_one_out_advantages(
        rewards[:, -1], prompts, reference_count=int(rewards.shape[0]),
    )
soft_q = beta * discount_rewards(terminal_advantages, rewards.shape[1], 0.98)
soft_q = beta * sigmoid_discounting(terminal_advantages, num_steps=50, k=7)

In [ ]:
rewards

In [ ]:
soft_q

In [ ]:
soft_q

In [ ]:
sample = torch.zeros_like(torch.randn(1, 50))
# terminal_rewards = torch.tensor([[[0.34], [0.45]]])
# terminal_rewards[:, None].expand(-1, len(range(0, 50)))
sample[:, -1] = 0.34
# 1 - gamma = 
effective_horizon = 50
# gamma = 1 - 1 / (effective_horizon)
gamma = 0.999
print(gamma)
for t in reversed(range(sample.shape[1] - 1)):
    sample[:, t] += sample[:, t + 1]*gamma**(sample.shape[1] - t - 1)
    
for s in sample:
    plt.plot(s.numpy())
    
plt.plot(sample[0].numpy()), sample.mean(), sample

In [ ]:
_means = trajectories[32]['action'].mean(axis=[2,3,4])
for m in _means[:]:
    plt.plot(m)
plt.show()
    
# gamma = 0.99
# rewards = trajectories[32]['rewards'].clone()
# for reward in rewards:
#     for t in reversed(range(reward.shape[0] - 1)):
#         reward[t+1] += reward[t + 1]*gamma**(reward.shape[0] - t - 1)
# gamma = 0.7  # sqrt(x) shape

# rewards = trajectories[32]['rewards'].clone()

# for reward in rewards:
#     T = reward.shape[0]
#     final_reward = reward[-1].clone()

#     for t in range(T):
#         x = t / (T - 1)  # goes from 0 to 1
#         reward[t] = final_reward * (x ** gamma)
k = 5.0  # sigmoid steepness; larger = sharper transition

rewards = trajectories[32]['rewards'].clone()

for reward in rewards:
    T = reward.shape[0]
    final_reward = reward[-1].clone()

    x = torch.linspace(
        0.0, 1.0, T,
        device=reward.device,
        dtype=reward.dtype
    )

    # Sigmoid centered at x=0.5
    s = torch.sigmoid(k * (x - 0.5))

    # Normalize so first value = 0 and last value = 1
    s = (s - s[0]) / (s[-1] - s[0])

    reward[:] = final_reward * s
for r in rewards:
    plt.plot(r.numpy())
plt.show()
# rewards[:, t+1]#*gamma**(rewards.shape[1] - t - 1)

In [ ]:
reward

In [ ]:
reference_rollout = trajectories.get(1)
# for i in range(1, 2):
    # rollout = trajectories.get(i)
combined_rollout = md3po_combined_rollouts(
    reference_rollout=reference_rollout,
    trajectories=trajectories
)
    

In [ ]:
combined_rollout['states'].shape

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

def _to_rgba_uint8(frame_chw):
    """Convert a single frame from CHW tensor/array to HWC uint8 RGBA."""
    frame = frame_chw.detach().cpu().numpy() if hasattr(frame_chw, "detach") else np.asarray(frame_chw)

    # CHW -> HWC
    if frame.ndim == 3 and frame.shape[0] in (1, 3, 4):
        frame = np.transpose(frame, (1, 2, 0))

    # Force 4 channels for consistent RGBA display
    if frame.ndim == 3 and frame.shape[2] == 1:
        frame = np.repeat(frame, 4, axis=2)
    elif frame.ndim == 3 and frame.shape[2] == 3:
        alpha = np.ones((frame.shape[0], frame.shape[1], 1), dtype=frame.dtype)
        frame = np.concatenate([frame, alpha], axis=2)

    # Normalize to uint8
    if np.issubdtype(frame.dtype, np.floating):
        fmin, fmax = float(frame.min()), float(frame.max())
        if fmax > fmin:
            frame = (frame - fmin) / (fmax - fmin)
        else:
            frame = np.zeros_like(frame)
        frame = (frame * 255).astype(np.uint8)
    elif frame.dtype != np.uint8:
        frame = np.clip(frame, 0, 255).astype(np.uint8)

    return frame


def plot_batch_single_row(batch_tensor):
    """Plot a batch of shape (N, C, H, W) in one row."""
    n = int(batch_tensor.shape[0])
    fig, axes = plt.subplots(1, n, figsize=(n * 1.4, 1.6), dpi=110)

    if n == 1:
        axes = [axes]

    for idx in range(n):
        rgba = _to_rgba_uint8(batch_tensor[idx])
        img = Image.fromarray(rgba, mode="RGBA")
        axes[idx].imshow(img)
        axes[idx].set_title(str(idx), fontsize=7, pad=2)
        axes[idx].axis("off")

    plt.subplots_adjust(wspace=0.02, hspace=0)
    plt.show()


batch = trajectories.get(1).get('state')[0]  # expected: (30, 4, 64, 64)
print("Batch shape:", tuple(batch.shape))
plot_batch_single_row(batch)

In [ ]:
import cv2
import torch
import numpy as np
from skimage.metrics import structural_similarity as ssim
from scipy.spatial.distance import cosine
from typing import List

# Function to calculate SSIM
def calculate_ssim(imageA, imageB):
    # Convert images to grayscale
    grayA = cv2.cvtColor(imageA, cv2.COLOR_BGR2GRAY)
    grayB = cv2.cvtColor(imageB, cv2.COLOR_BGR2GRAY)
    
    # Compute SSIM between the two images
    score, _ = ssim(grayA, grayB, full=True)
    return score

@torch.no_grad()
def clip_features(images: list[Image.Image], device: torch.device | str | None = None) -> List:
    """Compute CLIP features for a list of images.
    """
    import open_clip

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    
    # Load CLIP model
    model, _, preprocess = open_clip.create_model_and_transforms('ViT-H-14', pretrained='laion2b_s32b_b79k')
    model = model.to(device).eval()
    tokenizer = open_clip.get_tokenizer('ViT-H-14')
    
    # Preprocess images
    processed_images = torch.stack([
        preprocess(image.convert("RGB")) for image in images
    ]).to(device)
    
    
    # Compute features
    autocast_device = "cuda" if device == "cuda" or str(device).startswith("cuda") else "cpu"
    with torch.autocast(autocast_device, enabled=autocast_device == "cuda"):
        image_features = model.encode_image(processed_images)
    
    # Normalize and compute similarity
    image_features /= image_features.norm(dim=-1, keepdim=True)
    
    return image_features.cpu()

def plot_image(img1, img2):
    pil_img1 = Image.fromarray(img1)
    pil_img2 = Image.fromarray(img2)

    fig, axes = plt.subplots(1, 2, figsize=(5, 5))

    axes[0].imshow(pil_img1)
    axes[0].axis("off")

    axes[1].imshow(pil_img2)
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
combined_rollout['images'].shape

In [ ]:
trajectories[1]['images'].shape

In [ ]:
img = final_rollout_states.get(1)["final_state_tensor"]
reference_prompt = final_rollout_states.get(1)["prompt"]
reference_image = final_state_images[0]

for i in final_state_images[1:]:
    similarity = calculate_ssim(reference_image, )
    print(f"SSIM between the reference image and the current image: {similarity:.4f}")
    plot_image(reference_image, i)

In [ ]:
image_features = clip_features([Image.fromarray(img1), Image.fromarray(img2)])
similarity = 1 - cosine(image_features[0], image_features[1])
print(f"CLIP cosine similarity between the two images: {similarity:.4f}")

In [ ]:
trajectories.keys()